In [2]:

import csv
import os

def extract_domains(input_file):
    # Create output file names
    base_name = os.path.splitext(input_file)[0]
    output_file = f"{base_name}_extracted_domains.tsv"
    
    results = []
    
    with open(input_file, 'r') as f:
        # Read as tab-separated file
        reader = csv.reader(f, delimiter='\t')
        # Get header
        header = next(reader)
        
        # Process each row
        for row in reader:
            if len(row) < 10:  # Skip rows without enough columns
                continue
                
            # Skip rows with "NOT_FOUND" or "-" in terminal domains
            if row[4] == "-" or row[7] == "NOT_FOUND" or row[8] == "NOT_FOUND":
                continue
                
            species = row[2]
            common_name = row[3]
            protein = row[4]
            uniprot = row[5]
            
            # Get domain coordinates
            try:
                n_terminal_coords = row[7]
                c_terminal_coords = row[8]
                full_seq = row[9]
                
                # Extract N-terminal domain
                if "-" in n_terminal_coords:
                    start, end = map(int, n_terminal_coords.split("-"))
                    # Adjust for 0-based indexing in Python
                    n_terminal_seq = full_seq[start-1:end]
                else:
                    n_terminal_seq = ""
                
                # Extract C-terminal domain
                if "-" in c_terminal_coords:
                    start, end = map(int, c_terminal_coords.split("-"))
                    # Adjust for 0-based indexing in Python
                    c_terminal_seq = full_seq[start-1:end]
                else:
                    c_terminal_seq = ""
                
                # Add to results
                if n_terminal_seq or c_terminal_seq:
                    results.append({
                        'species': species,
                        'common_name': common_name,
                        'protein': protein,
                        'uniprot': uniprot,
                        'n_terminal_coords': n_terminal_coords,
                        'n_terminal_seq': n_terminal_seq,
                        'c_terminal_coords': c_terminal_coords,
                        'c_terminal_seq': c_terminal_seq
                    })
            except (ValueError, IndexError):
                print(f"Error processing row for {species} {protein}")
    
    # Write results to file
    with open(output_file, 'w') as out_f:
        writer = csv.writer(out_f, delimiter='\t')
        # Write header
        writer.writerow(['Species', 'Common Name', 'Protein', 'Uniprot ID', 
                         'N-terminal Coordinates', 'N-terminal Sequence',
                         'C-terminal Coordinates', 'C-terminal Sequence'])
        
        # Write data
        for item in results:
            writer.writerow([
                item['species'],
                item['common_name'],
                item['protein'],
                item['uniprot'],
                item['n_terminal_coords'],
                item['n_terminal_seq'],
                item['c_terminal_coords'],
                item['c_terminal_seq']
            ])
    
    print(f"Extracted domains written to {output_file}")
    return results

if __name__ == "__main__":
    input_file = "/home/ilnitsky/NPM/45_Species/Zoopark_Domains_NPM.tsv"
    extract_domains(input_file)

Extracted domains written to /home/ilnitsky/NPM/45_Species/Zoopark_Domains_NPM_extracted_domains.tsv
